In [4]:
#Imports
import pandas as pd
import numpy as np
import joblib
import smtplib
from email.message import EmailMessage

In [5]:
#Load the trained model and the feature schema
rf_model = joblib.load(r"F:\3000 stuff\random_forest_unsw_nb15.pkl")
feature_cols = joblib.load(r"F:\3000 stuff\rf_feature_columnsv2.pkl")

In [23]:
#Setting up Email Credentials and Alert
sender_email = "artemisids3000@gmail.com"
sender_password = "jihpygxwrmxthtfy"
recipient_email = "samuelstevens2902@gmail.com"

def send_email_alert(suspicious_events, threshold):
    if suspicious_events.empty:
        print("No high-risk event. No Email has been sent.")
        return
    #Set up email content
    msg = EmailMessage()
    msg["Subject"] = f"Artermis Alert: {len(suspicious_events)} High-risk events discovered"
    msg["From"] = sender_email
    msg["To"] = recipient_email

    #Make a summary of the top alerts
    alert_summary = suspicious_events[["id", "proto", "service", "attack_cat", "risk_score"]].head(10)

    body = f"""
    Suspicious Activity has been detected

    Threshold: {threshold}
    Suspicious events detected: {len(suspicious_events)}

    These are a number of the events that triggered this alert.
    {alert_summary.to_string(index=False)}

    This is an automated alert sent from Artemis IDS.
    """
    msg.set_content(body)

    #Sending the email
    try:
        with smtplib.SMTP_SSL("smtp.gmail.com", 465) as smtp:
            smtp.login(sender_email, sender_password)
            smtp.send_message(msg)
        print("Email sent successfully")
    except Exception as e:
        print("Failed to send Email", e)
    

In [7]:
#Load Test dataset
test_path = r"F:\3000 stuff\UNSW_NB15_testing-set.csv"
df_test = pd.read_csv(test_path).drop_duplicates().reset_index(drop=True)
df_test.head

<bound method NDFrame.head of           id       dur proto service state  spkts  dpkts  sbytes  dbytes  \
0          1  0.000011   udp       -   INT      2      0     496       0   
1          2  0.000008   udp       -   INT      2      0    1762       0   
2          3  0.000005   udp       -   INT      2      0    1068       0   
3          4  0.000006   udp       -   INT      2      0     900       0   
4          5  0.000010   udp       -   INT      2      0    2126       0   
...      ...       ...   ...     ...   ...    ...    ...     ...     ...   
82327  82328  0.000005   udp       -   INT      2      0     104       0   
82328  82329  1.106101   tcp       -   FIN     20      8   18062     354   
82329  82330  0.000000   arp       -   INT      1      0      46       0   
82330  82331  0.000000   arp       -   INT      1      0      46       0   
82331  82332  0.000009   udp       -   INT      2      0     104       0   

                rate  ...  ct_dst_sport_ltm  ct_dst_src_l

In [8]:
#Preserving certain columns in order to make text output more understandable
output_context = df_test[[ "state", "service", "proto", "attack_cat", "id" ]].copy()

In [9]:
#Drop non feature columns and columns that could produce leaks
drop_columns = ["srcip", "dstip", "attack_cat", "id"]
df_test = df_test.drop(
    columns=[c for c in drop_columns if c in df_test.columns],
    errors="ignore"
)

In [10]:
#Split the labels and the features
Y_test = df_test["label"].astype(int)
X_test = df_test.drop(columns=["label"])

In [11]:
#Data preprocessing and encoding
num_cols = X_test.select_dtypes(include=[np.number]).columns.tolist()
for col in num_cols:
    med = X_test[col].median()
    X_test[col] = X_test[col].fillna(med).replace([np.inf, -np.inf], med)

cat_cols = X_test.select_dtypes(include=["object"]).columns.tolist()
X_test_enc = pd.get_dummies(X_test, columns=cat_cols, drop_first="False")


In [12]:
#Feature allignment to ensure to mismatch between the columns of test and training

X_test_enc = X_test_enc.reindex(columns=feature_cols, fill_value=0)


X_test_enc = X_test_enc.astype("float64")



In [13]:
#Ensure that the model can run despite columns being missing betweeen training and testing
print("Extra columns:", set(X_test_enc.columns) - set(feature_cols))
print("Missing columns:", set(feature_cols) - set(X_test_enc.columns))
print("Shape:", X_test_enc.shape)


Extra columns: set()
Missing columns: set()
Shape: (82332, 194)


In [14]:
#Prediction of the possiblity of suspicious actions
probs = rf_model.predict_proba(X_test_enc)[:, 1]


In [15]:
#Set and apply detection threshold
threshold = 0.9

results = output_context.copy()
results["risk_score"] = probs
results["suspicious"] = probs >= threshold

In [24]:
#Extract events marked as suspicious and then write to a TXT file
suspicious_events = results[results["suspicious"]]
output_file = r"F:\3000 stuff\log_events_marked_by_rf.txt"

with open(output_file, "w") as f:
    for _, row in suspicious_events.iterrows():
        f.write(
            f"[ID={row['id']}] "
            f"[RISK={row['risk_score']:.2f}] "
            f"PROTO={row['proto']} SERVICE={row['service']} "
            f"ATTACK={row['attack_cat']}\n"
)

print(f"Data Extraction Complete. {len(suspicious_events)} suspicous events written to:")
print(output_file)
send_email_alert(suspicious_events, threshold)



Data Extraction Complete. 37484 suspicous events written to:
F:\3000 stuff\log_events_marked_by_rf.txt
Email sent successfully
